# Partie 3 — UDF Python, expressions natives et stockage Parquet

**Application :** `Haitam-Partie3-UDF-Parquet`  
**Environnement :** Spark 3.5.8, cluster Standalone, MinIO / S3A  
**Sources :** historique Bronze `orders_history.jsonl` et référentiel restaurants `restaurants.json`.

## Objectifs

1. Charger et profiler les événements Bronze.
2. Classifier `distance_km` avec une UDF Python puis avec les expressions natives Spark.
3. Vérifier l'équivalence exacte des deux implémentations et tester les valeurs limites.
4. Comparer leurs performances avec un benchmark échauffé et alterné.
5. Écrire et relire un Parquet de test partitionné par ville.
6. Enrichir les commandes avec le référentiel restaurants et produire le dataset Silver final `s3a://lake/silver/orders_history/`.

> **Prérequis d'exécution :** Docker Compose doit être démarré ; le Master et au moins un Worker Spark doivent être disponibles ; MinIO/S3A doit être configuré dans l'image Spark. Si Spark indique `Initial job has not accepted any resources`, libérer les cœurs utilisés par une autre application dans Spark Master UI (`http://localhost:8080`) avant de relancer les cellules.

> **Important :** le dossier `orders_history_udf_test/` est un emplacement de test séparé. Le dataset final Silver n'est jamais écrasé : si `orders_history/` existe déjà, le notebook le relit et vérifie son contenu.

## 1. Initialisation de la session Spark

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("Haitam-Partie3-UDF-Parquet")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("========== SPARK SESSION ==========")
print("Spark version :", spark.version)
print("Application  :", sc.appName)
print("Master       :", sc.master)
print("Parallelism  :", sc.defaultParallelism)


## 2. Chargement des événements Bronze

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType, DecimalType, TimestampType
)

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("restaurant_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("status", StringType(), True),
    StructField("amount_mad", DecimalType(12, 2), True),
    StructField("items_count", IntegerType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("courier_id", StringType(), True),
    StructField("event_time", TimestampType(), True),
])

orders_path = "s3a://lake/bronze/history/orders_history.jsonl"

df_orders = (
    spark.read
    .schema(orders_schema)
    .option("mode", "FAILFAST")
    .json(orders_path)
)

orders_count = df_orders.count()
print("Événements chargés :", orders_count)
assert orders_count == 50_000, f"Nombre inattendu d'événements : {orders_count}"
print("ORDERS RELOADED SUCCESSFULLY")

df_orders.printSchema()
df_orders.show(5, truncate=False)


## 3. Profil des distances et répartition initiale

In [ ]:
from pyspark.sql import functions as F

distance_profile = df_orders.agg(
    F.count("*").alias("total_events"),
    F.count("distance_km").alias("non_null_distances"),
    F.sum(F.when(F.col("distance_km").isNull(), 1).otherwise(0)).alias("null_distances"),
    F.sum(F.when(F.isnan("distance_km"), 1).otherwise(0)).alias("nan_distances"),
    F.min("distance_km").alias("min_distance"),
    F.max("distance_km").alias("max_distance"),
    F.avg("distance_km").alias("average_distance"),
    F.sum(F.when(F.col("distance_km") < 0, 1).otherwise(0)).alias("negative_distances")
)

print("========== DISTANCE PROFILE ==========")
distance_profile.show(truncate=False)

distance_distribution = (
    df_orders
    .withColumn(
        "distance_category",
        F.when(F.col("distance_km").isNull(), "NULL")
         .when(F.isnan("distance_km"), "INVALID")
         .when(F.col("distance_km") < 2, "court")
         .when(F.col("distance_km") <= 5, "moyen")
         .otherwise("long")
    )
    .groupBy("distance_category")
    .count()
    .orderBy("distance_category")
)

print("========== DISTANCE DISTRIBUTION ==========")
distance_distribution.show(truncate=False)


## 4. Implémentation avec une UDF Python

Règles de classification :

- distance strictement inférieure à 2 km : `court` ;
- distance comprise entre 2 et 5 km inclus : `moyen` ;
- distance strictement supérieure à 5 km : `long` ;
- valeur nulle ou `NaN` : `NULL` (valeur Python `None`).

In [ ]:
import math
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType


def distance_band(distance):
    if distance is None or math.isnan(distance):
        return None
    if distance < 2:
        return "court"
    if distance <= 5:
        return "moyen"
    return "long"


distance_band_udf = udf(distance_band, StringType())

df_udf = df_orders.withColumn(
    "distance_category",
    distance_band_udf(col("distance_km"))
)

print("========== UDF PYTHON ==========")
df_udf.select("order_id", "distance_km", "distance_category").show(10, truncate=False)

df_udf.groupBy("distance_category").count().orderBy("distance_category").show()


## 5. Implémentation avec les expressions natives Spark

In [ ]:
distance = F.col("distance_km")
invalid_distance = (
    distance.isNull()
    | F.coalesce(F.isnan(distance), F.lit(False))
)

df_native = df_orders.withColumn(
    "distance_category",
    F.when(invalid_distance, F.lit(None).cast("string"))
     .when(distance < 2, "court")
     .when(distance <= 5, "moyen")
     .otherwise("long")
)

print("========== NATIVE SPARK FUNCTIONS ==========")
df_native.select("order_id", "distance_km", "distance_category").show(10, truncate=False)

df_native.groupBy("distance_category").count().orderBy("distance_category").show()


## 6. Vérification de l’équivalence UDF / Native Spark

In [ ]:
udf_result = df_udf.select("order_id", "distance_km", "distance_category")
native_result = df_native.select("order_id", "distance_km", "distance_category")

udf_only_count = udf_result.exceptAll(native_result).count()
native_only_count = native_result.exceptAll(udf_result).count()

total_mismatches = udf_only_count + native_only_count

print("========== EQUIVALENCE TEST ==========")
print("UDF uniquement    :", udf_only_count)
print("Native uniquement :", native_only_count)
print("Total différences :", total_mismatches)

assert total_mismatches == 0, "Les deux implémentations produisent des résultats différents."
print("SUCCESS : les deux méthodes produisent exactement les mêmes résultats.")


## 7. Tests des valeurs limites

In [ ]:
test_schema = StructType([
    StructField("distance_km", DoubleType(), True)
])

test_values = [
    (0.5,),
    (1.99,),
    (2.0,),
    (5.0,),
    (5.01,),
    (20.0,),
    (None,),
    (float("nan"),),
]

df_distance_tests = spark.createDataFrame(test_values, schema=test_schema)

test_udf = df_distance_tests.withColumn(
    "category", distance_band_udf(F.col("distance_km"))
)

test_native = df_distance_tests.withColumn(
    "category",
    F.when(
        F.col("distance_km").isNull()
        | F.coalesce(F.isnan(F.col("distance_km")), F.lit(False)),
        F.lit(None).cast("string")
    )
    .when(F.col("distance_km") < 2, "court")
    .when(F.col("distance_km") <= 5, "moyen")
    .otherwise("long")
)

print("========== UDF BOUNDARY TESTS ==========")
test_udf.orderBy("distance_km").show(truncate=False)

print("========== NATIVE BOUNDARY TESTS ==========")
test_native.orderBy("distance_km").show(truncate=False)

udf_diff = test_udf.exceptAll(test_native).count()
native_diff = test_native.exceptAll(test_udf).count()

print("UDF differences    :", udf_diff)
print("Native differences :", native_diff)
assert udf_diff == 0 and native_diff == 0
print("SUCCESS : all boundary tests passed.")


## 8. Benchmark UDF Python / Native Spark

Le benchmark met en cache les données d’entrée, effectue trois cycles d’échauffement, puis dix mesures par méthode. L’ordre est alterné pour limiter le biais lié à l’ordre d’exécution. La médiane est utilisée pour comparer les temps.

**Cette cellule est coûteuse :** elle lance plusieurs actions Spark. Elle peut être ignorée si les mesures finales ont déjà été obtenues et enregistrées dans le compte rendu du projet.

In [ ]:
from pyspark import StorageLevel
import time
import statistics

# Mettre les données d'entrée en cache et forcer leur chargement.
df_orders.persist(StorageLevel.MEMORY_AND_DISK)
cached_count = df_orders.count()
print("Événements mis en cache :", cached_count)
print("Storage level :", df_orders.storageLevel)


def calculate_categories(dataframe):
    return dataframe.groupBy("distance_category").count().collect()


# Échauffement.
print("========== WARM-UP ==========")
for i in range(1, 4):
    calculate_categories(df_udf)
    calculate_categories(df_native)
    print(f"Warm-up {i}/3 terminé")


# Dix mesures par méthode, avec alternance de l'ordre.
benchmark_results = {
    "UDF Python": [],
    "Native Spark": []
}

print("========== BENCHMARK EXECUTIONS ==========")
for trial in range(1, 11):
    methods = (
        ["UDF Python", "Native Spark"]
        if trial % 2 == 1
        else ["Native Spark", "UDF Python"]
    )

    for method in methods:
        start_time = time.perf_counter()
        if method == "UDF Python":
            calculate_categories(df_udf)
        else:
            calculate_categories(df_native)
        elapsed = time.perf_counter() - start_time
        benchmark_results[method].append(elapsed)
        print(f"Essai {trial:02d} | {method:12s} | {elapsed:.4f} s")


print("========== FINAL BENCHMARK ==========")
for method, values in benchmark_results.items():
    print(f"\n{method}")
    print(f"Moyenne : {statistics.mean(values):.4f} s")
    print(f"Médiane : {statistics.median(values):.4f} s")
    print(f"Min     : {min(values):.4f} s")
    print(f"Max     : {max(values):.4f} s")

udf_median = statistics.median(benchmark_results["UDF Python"])
native_median = statistics.median(benchmark_results["Native Spark"])

print("\n========== PERFORMANCE COMPARISON ==========")
print(f"UDF Python median    : {udf_median:.4f} s")
print(f"Native Spark median  : {native_median:.4f} s")
if native_median > 0:
    print(f"Speedup UDF / Native : {udf_median / native_median:.2f}x")

# Libérer le cache après les mesures.
df_orders.unpersist()


## 9. Écriture et validation du Parquet de test

Cette première écriture valide la classification sur le DataFrame de commandes seul. Elle utilise un dossier de test distinct du Silver enrichi final.

In [ ]:
parquet_test_path = "s3a://lake/silver/orders_history_udf_test/"

print("========== PARQUET TEST WRITE ==========")
(
    df_native.write
    .mode("overwrite")
    .partitionBy("city")
    .parquet(parquet_test_path)
)
print("Écriture terminée :", parquet_test_path)

# Relecture.
df_parquet = spark.read.parquet(parquet_test_path)
print("========== PARQUET TEST SCHEMA ==========")
df_parquet.printSchema()

parquet_count = df_parquet.count()
category_counts = {
    row["distance_category"]: row["count"]
    for row in df_parquet.groupBy("distance_category").count().collect()
}
city_count = df_parquet.select("city").distinct().count()

comparison_cols = ["order_id", "distance_km", "distance_category", "city"]
before_write = df_native.select(*comparison_cols)
after_read = df_parquet.select(*comparison_cols)
missing_after_read = before_write.exceptAll(after_read).count()
unexpected_after_read = after_read.exceptAll(before_write).count()

print("Nombre de lignes        :", parquet_count)
print("Répartition catégories  :", category_counts)
print("Nombre de villes        :", city_count)
print("Lignes manquantes       :", missing_after_read)
print("Lignes inattendues      :", unexpected_after_read)

assert parquet_count == 50_000
assert category_counts == {"court": 3_800, "moyen": 7_604, "long": 38_596}
assert city_count == 10
assert missing_after_read == 0
assert unexpected_after_read == 0
print("SUCCESS : Parquet test validation passed.")


## 10. Intégration du référentiel restaurants et Silver final

Cette étape reconstruit le dataset enrichi à partir des deux sources Bronze/Reference, ajoute la catégorie de distance avec les expressions natives Spark, puis écrit ou relit le dataset Silver final.

Le code est idempotent du point de vue de la destination : si le chemin final existe déjà, aucune écriture n'est relancée. Le contenu existant est comparé au résultat reconstruit ; toute différence déclenche une assertion.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DecimalType
from decimal import Decimal

# 1. Préparer les commandes avec les types attendus.
orders_for_silver = df_orders.select(
    F.col("order_id").cast("string"),
    F.col("restaurant_id").cast("string"),
    F.col("customer_id").cast("string"),
    F.col("city").cast("string"),
    F.col("status").cast("string"),
    F.col("amount_mad").cast(DecimalType(12, 2)),
    F.col("items_count").cast("int"),
    F.col("distance_km").cast("double"),
    F.col("courier_id").cast("string"),
    F.to_timestamp("event_time").alias("event_time")
)

# 2. Lire le référentiel restaurants.
restaurants_schema = StructType([
    StructField("restaurant_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("category", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True)
])

restaurants_path = "s3a://lake/reference/restaurants.json"
restaurants_df = (
    spark.read
    .schema(restaurants_schema)
    .option("multiLine", True)
    .option("mode", "FAILFAST")
    .json(restaurants_path)
)

orders_count = orders_for_silver.count()
restaurants_count = restaurants_df.count()
restaurant_id_duplicates = (
    restaurants_df.groupBy("restaurant_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Événements :", orders_count)
print("Restaurants :", restaurants_count)
print("Identifiants restaurants dupliqués :", restaurant_id_duplicates)

assert orders_count == 50_000
assert restaurants_count == 200
assert restaurant_id_duplicates == 0

# 3. Vérifier que chaque commande référence un restaurant connu.
restaurant_lookup = restaurants_df.select(
    "restaurant_id",
    F.col("name").alias("restaurant_name"),
    F.col("city").alias("restaurant_city"),
    "category",
    "lat",
    "lon"
)

unmatched_count = (
    orders_for_silver
    .join(restaurant_lookup.select("restaurant_id"), on="restaurant_id", how="left_anti")
    .count()
)
print("Commandes sans restaurant correspondant :", unmatched_count)
assert unmatched_count == 0

# 4. Jointure et validation de l'enrichissement.
enriched_df = orders_for_silver.join(
    restaurant_lookup,
    on="restaurant_id",
    how="inner"
)
enriched_count = enriched_df.count()
print("Lignes après enrichissement :", enriched_count)
assert enriched_count == 50_000

missing_restaurant_attributes = enriched_df.filter(
    F.col("restaurant_name").isNull() | F.col("category").isNull()
).count()
print("Attributs restaurant manquants :", missing_restaurant_attributes)
assert missing_restaurant_attributes == 0

# 5. Comparer UDF et expressions natives sur le DataFrame enrichi.
distance = F.col("distance_km")
enriched_udf_df = enriched_df.withColumn(
    "distance_category",
    distance_band_udf(distance)
)

enriched_native_df = enriched_df.withColumn(
    "distance_category",
    F.when(
        distance.isNull() | F.coalesce(F.isnan(distance), F.lit(False)),
        F.lit(None).cast("string")
    )
    .when(distance < 2, "court")
    .when(distance <= 5, "moyen")
    .otherwise("long")
)

udf_only = enriched_udf_df.exceptAll(enriched_native_df).count()
native_only = enriched_native_df.exceptAll(enriched_udf_df).count()
print("Différences UDF -> Native :", udf_only)
print("Différences Native -> UDF :", native_only)
assert udf_only == 0 and native_only == 0

# 6. Écrire le Silver final uniquement si la destination n'existe pas.
output_path = "s3a://lake/silver/orders_history/"
path = spark._jvm.org.apache.hadoop.fs.Path(output_path)
fs = path.getFileSystem(spark._jsc.hadoopConfiguration())

if fs.exists(path):
    print("Destination Silver déjà présente : aucune réécriture.")
else:
    (
        enriched_native_df.write
        .mode("errorifexists")
        .partitionBy("city")
        .parquet(output_path)
    )
    print("Dataset Silver écrit :", output_path)

# 7. Relire et comparer exactement le dataset final.
silver_df = spark.read.parquet(output_path)
columns = enriched_native_df.columns
source = enriched_native_df.select(*columns)
reloaded = silver_df.select(*columns)

silver_count = silver_df.count()
city_count = silver_df.select("city").distinct().count()
missing_enrichment = silver_df.filter(
    F.col("restaurant_name").isNull()
    | F.col("category").isNull()
    | F.col("distance_category").isNull()
).count()

diff_source_to_silver = source.exceptAll(reloaded).count()
diff_silver_to_source = reloaded.exceptAll(source).count()

print("========== SILVER VALIDATION ==========")
print("Lignes Silver                  :", silver_count)
print("Villes                         :", city_count)
print("Données enrichies manquantes   :", missing_enrichment)
print("Différences source -> Silver   :", diff_source_to_silver)
print("Différences Silver -> source   :", diff_silver_to_source)

assert silver_count == 50_000
assert city_count == 10
assert missing_enrichment == 0
assert diff_source_to_silver == 0
assert diff_silver_to_source == 0

# 8. Contrôles métiers des événements livrés.
delivered_df = silver_df.filter(F.col("status") == "DELIVERED")
delivered_metrics = delivered_df.agg(
    F.count("*").alias("delivered_events"),
    F.sum("amount_mad").alias("revenue_mad")
).first()

delivered_count = delivered_metrics["delivered_events"]
delivered_revenue = delivered_metrics["revenue_mad"]
city_category_groups = (
    delivered_df.groupBy("city", "category").count().count()
)

print("Livraisons DELIVERED            :", delivered_count)
print("Chiffre d'affaires (MAD)        :", delivered_revenue)
print("Groupes ville/catégorie         :", city_category_groups)

assert delivered_count == 27_360
assert delivered_revenue == Decimal("6514937.82")
assert city_category_groups == 70

print("VALIDATION SILVER : SUCCESS")

silver_df.select(
    "order_id", "city", "restaurant_name", "category",
    "distance_km", "distance_category"
).show(10, truncate=False)


## 11. Bilan de la Partie 3

À la fin d'une exécution complète, les contrôles attendus sont :

- **Source Bronze :** 50 000 événements.
- **Référentiel :** 200 restaurants et aucun identifiant dupliqué.
- **Classification :** égalité exacte entre UDF Python et expressions natives Spark, y compris les valeurs limites.
- **Benchmark :** dix mesures par approche après échauffement ; comparer les médianes réellement mesurées, sans supposer à l'avance quelle approche gagne.
- **Parquet de test :** 50 000 lignes, dix villes et zéro différence après relecture.
- **Silver enrichi :** 50 000 lignes, dix villes, zéro donnée d'enrichissement manquante et zéro différence avant/après stockage.
- **Contrôles métiers :** 27 360 événements `DELIVERED`, 6 514 937,82 MAD et 70 groupes ville/catégorie.

**Interprétation :** les expressions natives Spark sont la méthode retenue pour le dataset Silver. La UDF Python est conservée pour la comparaison fonctionnelle et le benchmark.

> Les cellules de ce fichier sont livrables et prêtes à être exécutées, mais les sorties d'exécution du fichier source incomplet n'ont pas été fabriquées. Exécuter les cellules sur le cluster du projet pour enregistrer les nouveaux résultats dans le notebook avant le commit final.